# Beyond CUSUM: likelihood, Bayesian inference and decision costs

Classical CUSUM is an appropriate baseline, not the endpoint of change-point research. This experiment compares a one-sided CUSUM, Gaussian generalized likelihood-ratio (GLR) scan, and conjugate-normal Bayesian online change-point detection (BOCPD) against frozen and scheduled model maintenance.

All three detectors observe the *same* standardised, delayed univariate score measuring disagreement with a frozen regression model. Null thresholds are calibrated on independent no-change simulation streams, not selected from evaluation results.

## Statistical hypotheses

CUSUM accumulates one-sided evidence:

\[
G_t=\max(0,G_{t-1}+Z_t-\kappa).
\]

GLR tests an unknown positive mean shift over each recent suffix length \(m\):

\[
\operatorname{GLR}_t=\max_{m\in[m_{\min},m_{\max}]}
\frac{(\max(0,\sum_{j=t-m+1}^t Z_j))^2}{2m}.
\]

The Bayesian detector maintains the posterior distribution over the current segment run length \(r_t\) with a geometric hazard \(h\), a normal prior for the unknown segment mean and a unit Gaussian observation variance. The alarm score is the posterior probability of a short run length following an initial warmup:

\[
A_t=P(r_t<w\mid Z_{1:t}).
\]

These probabilities are conditional on the assumed Bayesian model, which may be misspecified under AR(1) dependence.

The null calibration uses the maximum score attained across the entire fixed evaluation horizon. An independent rank-corrected null threshold is selected for each detector, response mechanism, feedback delay and noise-dependence family. It is not an anytime-valid sequential test.

In [ ]:
from __future__ import annotations
import os
import matplotlib.pyplot as plt
from IPython.display import display
from comparative_change_detection import (
    ComparisonDesign, run_comparison, summarize_comparison,
)

SMOKE: bool = os.environ.get("COMPARATIVE_SMOKE", "0") == "1"
REPEATS: int = int(os.environ.get("COMPARATIVE_REPEATS", "6"))
if REPEATS < 1:
    raise ValueError("COMPARATIVE_REPEATS must be positive")

design = ComparisonDesign(
    n_fit=80 if SMOKE else 160,
    n_cal=35 if SMOKE else 80,
    n_live=45 if SMOKE else 120,
    null_streams=19 if SMOKE else 99,
    delays=(0,) if SMOKE else (0, 12),
    scheduled_every=10 if SMOKE else 15,
    refit_window=70 if SMOKE else 100,
    glr_max_window=16 if SMOKE else 32,
    action_costs=(0.6,) if SMOKE else (0.2, 0.6, 1.0, 1.4),
    refit_costs=(0.0, 0.2) if SMOKE else (0.0, 0.05, 0.25),
)

calibration, models, decisions = run_comparison(
    design,
    evaluation_repeats=REPEATS,
    scenarios=("low_variance_signal",) if SMOKE
        else ("low_variance_signal", "high_variance_signal"),
    regimes=("stable", "gradual_drift") if SMOKE
        else ("stable", "gradual_drift", "ar1", "gradual_ar1"),
)
operating, economic = summarize_comparison(models, decisions)
display(calibration.round(3))
display(operating.round(3))
display(economic.head(15).round(3))

## What constitutes a good detector?

Detection quality is not just whether an alarm is raised. Stable regimes measure the probability of at least one false alarm. Gradually drifting regimes offer a defined start for reporting descriptive detection latency, but unlike an abrupt change there is no unique optimal onset. We also count retraining events and evaluate test-stream RMSE and net economic regret, including each retraining charge.

We compare detectors and economic policies on *paired streams*; observations, costs and alarm events within a stream are dependent and do not count as independent Monte Carlo replications.

In [ ]:
focus = economic.loc[
    (economic["scenario"] == "low_variance_signal")
    & (economic["regime"] == "gradual_drift")
    & (economic["delay"] == 0)
    & (economic["action_cost"] == 0.6)
]
fig, ax = plt.subplots(figsize=(9, 4))
for policy, group in focus.groupby("policy"):
    group = group.sort_values("refit_cost")
    ax.plot(group["refit_cost"], group["mean_net_regret"], marker="o", label=policy)
ax.set(xlabel="Maintenance cost per retraining",
       ylabel="Mean economic regret",
       title="Detection evidence versus model-maintenance economics")
ax.legend()
fig.tight_layout()
plt.show()
assert (decisions["net_regret"] >= -1e-9).all()

## Interpreting the comparison

No single method wins by definition. GLR assumes a Gaussian mean shift with known scale; BOCPD assumes a constant change hazard and a conjugate Gaussian mean model. All operate on one score, not the full predictor distribution. Results depend on the score definition, detector tuning, delayed labels, drift mechanism and retraining costs.

The null rank threshold offers only a *fixed-horizon marginal* false-alarm statement, assuming exchangeable whole-stream null maxima. On gradual drift, the reported latency is descriptive. Default Monte Carlo sample sizes are exploratory and no causal intervention effect has been identified.

Next, compare calibrated detector families under broader change mechanisms (variance shift, nonlinear relations and unknown noise dependence), pre-register a common cost objective, and increase independent simulation repetitions before claiming superiority.